# TT1 - MDM UBA - 2025

**Tariff classification using NLP**

New enviroment is needed for replication of doc2vec baseline

**doc2vec** & **fasttext** library

!pip install gensim==4.3.3

In [1]:
# Gral dependencies
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
from datetime import datetime
import re
from typing import Iterable, List, Tuple, Union, Optional, Dict, Any
from collections import defaultdict
import matplotlib.pyplot as plt

### Raw dataset

In [2]:
colspecs = [(0, 6), (6, None)]
data_type = {'HS06': str}
df = pd.read_fwf('data/raw_data_HScodes_desc.txt',
                 colspecs=colspecs, header=None,
                 names=['HS06', 'GOODS_DESCRIPTION'],
                 dtype=data_type)

### Quick EDA

null and duplicated samples

dropping duplicates

analyzing tops and bottoms regarding frequencies

In [3]:
# Quick EDA
print("=== Quick EDA ===")

# Add HS02 (chapter) and HS04 (heading)
df['HS04'] = df['HS06'].str[:4]
df['HS02'] = df['HS06'].str[:2]

print("Nulls per column:")
print(df.isnull().sum(), "\n")

print("Duplicate rows:", df.duplicated().sum(), "\n")

# Function to build and display freq tables
def freq_table(col, name):
    vc      = df[col].value_counts().rename('count')
    rel     = df[col].value_counts(normalize=True).rename('rel_freq')
    cum     = rel.cumsum().rename('cum_freq')
    summary = pd.concat([vc, rel, cum], axis=1)
    summary['rel_freq'] = (summary['rel_freq'] * 100).round(2).astype(str) + '%'
    summary['cum_freq'] = (summary['cum_freq'] * 100).round(2).astype(str) + '%'

    print(f"## Samples per {name} ({col})\n")
    print("### Top 10")
    print(summary.head(10).to_markdown(), "\n")
    print("### Bottom 10")
    print(summary.tail(10).to_markdown(), "\n")

# Dropping duplicates
df.drop_duplicates(inplace=True)

# Chapter-level (HS02)
freq_table('HS02', 'chapter')

# Heading-level (HS04)
freq_table('HS04', 'heading')

# Subheading-level (HS06)
freq_table('HS06', 'subheading')

=== Quick EDA ===
Nulls per column:
HS06                 0
GOODS_DESCRIPTION    0
HS04                 0
HS02                 0
dtype: int64 

Duplicate rows: 232220 

## Samples per chapter (HS02)

### Top 10
|   HS02 |   count | rel_freq   | cum_freq   |
|-------:|--------:|:-----------|:-----------|
|     84 |   54901 | 20.5%      | 20.5%      |
|     85 |   33571 | 12.54%     | 33.04%     |
|     87 |   28476 | 10.63%     | 43.67%     |
|     73 |   16173 | 6.04%      | 49.71%     |
|     39 |   12218 | 4.56%      | 54.28%     |
|     90 |   11611 | 4.34%      | 58.61%     |
|     82 |    7972 | 2.98%      | 61.59%     |
|     94 |    7921 | 2.96%      | 64.55%     |
|     40 |    7526 | 2.81%      | 67.36%     |
|     83 |    4285 | 1.6%       | 68.96%     | 

### Bottom 10
|   HS02 |   count | rel_freq   | cum_freq   |
|-------:|--------:|:-----------|:-----------|
|     41 |      22 | 0.01%      | 99.96%     |
|     81 |      19 | 0.01%      | 99.97%     |
|     45 |      19 | 0

### Preprocessing of text

In [4]:
stop_words = {'of', 'or', 'and', 'for', 'than', 'the', 'in', 'with', 'to', 'but', 'by'
             , 'whether', 'on', 'its', 'an', 'their', 'at', 'this', 'which', 'from'
             , 'as', 'be', 'is'}
alphabet_pattern = re.compile(r'[^a-zA-Z]')
alphabet_number_pattern = re.compile(r'[^a-zA-Z0-9]')
remove_pattern = re.compile(r'[\;\,\)\(\[\]\:]')


def refine_text_func(text):
    text = text.lower()
    text = ' '.join([w for w in text.split() if w not in stop_words])
    alphabet = re.sub(alphabet_pattern, ' ', text)
    alphabet_number = re.sub(alphabet_number_pattern, ' ', text)
    remove = re.sub(remove_pattern, ' ', text)
    result = ' '.join([text, alphabet, alphabet_number, remove])
    return result

In [5]:
df['PREPRO_DESCRIPTION'] = df['GOODS_DESCRIPTION'].progress_apply(lambda x: refine_text_func(x))

100%|██████████| 267780/267780 [00:01<00:00, 194966.26it/s]


### N-gram generation

In [6]:
def create_ngram_data(text, ngram_value=2):
    text_list = text.split()
    ngram_list = list(zip(*[text_list[i:] for i in range(ngram_value)]))
    result = []
    for n_data in ngram_list:
        result.append('_'.join(n_data))
    return ' '.join(result)

create_ngram_data('LIVE BREEDING FARM HORSE')

'LIVE_BREEDING BREEDING_FARM FARM_HORSE'

In [7]:
df['NGRAM_DESCRIPTION'] = df['PREPRO_DESCRIPTION'].progress_apply(lambda x: create_ngram_data(x))

100%|██████████| 267780/267780 [00:00<00:00, 279960.89it/s]


In [8]:
df.head()

,HS06,GOODS_DESCRIPTION,HS04,HS02,PREPRO_DESCRIPTION,NGRAM_DESCRIPTION
0,271019,BRAKE FLUID DOT 4 50X200ML,2710,27,brake fluid dot 4 50x200ml brake fluid dot ...,brake_fluid fluid_dot dot_4 4_50x200ml 50x200m...
1,847710,PLASTIC INJECTION MOULD MODEL 21A 110G DSM1010...,8477,84,plastic injection mould model 21a 110g dsm1010...,plastic_injection injection_mould mould_model ...
2,844399,LCD ASSEMBLY,8443,84,lcd assembly lcd assembly lcd assembly lcd ass...,lcd_assembly assembly_lcd lcd_assembly assembl...
3,848280,BEARING 22238 KCAW33C3 BRAND MCB,8482,84,bearing 22238 kcaw33c3 brand mcb bearing ...,bearing_22238 22238_kcaw33c3 kcaw33c3_brand br...
4,630900,USED HANDBAGS AND WALLETS,6309,63,used handbags wallets used handbags wallets us...,used_handbags handbags_wallets wallets_used us...


Sampling function

In [9]:
def bootstrap_sampling(df, test_fraction=0.1, seed=32):
    # Determine the number of test samples
    n_test = int(len(df) * test_fraction)
    # Perform bootstrap sampling for the test set
    test_set = df.sample(n=n_test, replace=True, random_state=seed)
    # Remove the test samples from the original dataframe to create the training set
    train_set = df.drop(test_set.index)
    
    return train_set, test_set

In [10]:
# Split the data into train and validation sets
train_df, val_df = bootstrap_sampling(df, test_fraction=0.1)

## Training a Doc2Vec as baseline

A- Raw descriptions

B- Preproced descriptions

C- Preproced + N-gram descriptions

In [11]:
# Model dependences
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec

target_col = 'HS04'
window = 5 # context window size +/- 
num_epochs = 50
model_dim = 254
seed = 32

raw_col = 'GOODS_DESCRIPTION'
prepro_col = 'PREPRO_DESCRIPTION'
ngram_col = 'NGRAM_DESCRIPTION'

#### A- Raw descriptions

In [12]:
model = Doc2Vec(window=window, 
                min_count=1, # ignore 1 instead of not ignore any words
                vector_size=model_dim, # dim of the feature vectors
                sample=1e-4, # threshold randomly down-sample high-frequency words
                hs=1, # hierarchical softmax instead of negative sampling
                max_vocab_size=None, # no limit
                alpha=0.025, # initial learning rate
                min_alpha=0.001, # min learning rate
                dm=0, # PV-DBOW 
                dbow_words=0, # only trains doc-vectors
                dm_tag_count=1, # one tag per document
                dm_mean=0, # use the sum of the context word vectors
                dm_concat=0, # for smaller model
                negative=5, # number of negative samples
                seed=32, # random seed
                workers=os.cpu_count()
                )

In [13]:
print("Building vocabulary...")

sentences = []
for idx, row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    words_list = row[raw_col].split()
    sentences.append(TaggedDocument(words_list, [row[target_col]]))

print(sentences[:3])

model.build_vocab(sentences)

Building vocabulary...


100%|██████████| 242289/242289 [00:04<00:00, 55052.22it/s]


[TaggedDocument(words=['BRAKE', 'FLUID', 'DOT', '4', '50X200ML'], tags=['2710']), TaggedDocument(words=['LCD', 'ASSEMBLY'], tags=['8443']), TaggedDocument(words=['USED', 'HANDBAGS', 'AND', 'WALLETS'], tags=['6309'])]


In [14]:
model.train(sentences, total_examples=model.corpus_count, epochs=num_epochs)

In [15]:
model_name = 'doc2vec_hs04_raw'
time_now = datetime.now().strftime('%d%m%Y_%H%M%S')

model_save_path = f'models/doc2vec/{model_name}_{num_epochs}epochs_{time_now}.d2v'
model.save(model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to models/doc2vec/doc2vec_hs04_raw_50epochs_16112025_100112.d2v


In [16]:
del model

#### B- Preproced descriptions

In [17]:
model = Doc2Vec(window=window, 
                min_count=1, # ignore 1 instead of not ignore any words
                vector_size=model_dim, # dim of the feature vectors
                sample=1e-4, # threshold randomly down-sample high-frequency words
                hs=1, # hierarchical softmax instead of negative sampling
                max_vocab_size=None, # no limit
                alpha=0.025, # initial learning rate
                min_alpha=0.001, # min learning rate
                dm=0, # PV-DBOW 
                dbow_words=0, # only trains doc-vectors
                dm_tag_count=1, # one tag per document
                dm_mean=0, # use the sum of the context word vectors
                dm_concat=0, # for smaller model
                negative=5, # number of negative samples
                seed=32, # random seed
                workers=os.cpu_count()
                )

In [18]:
print("Building vocabulary...")

sentences = []
for idx, row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    words_list = row[prepro_col].split()
    sentences.append(TaggedDocument(words_list, [row[target_col]]))

print(sentences[:3])

model.build_vocab(sentences)

Building vocabulary...


100%|██████████| 242289/242289 [00:04<00:00, 50666.97it/s]


[TaggedDocument(words=['brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', 'x', 'ml', 'brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', '4', '50x200ml'], tags=['2710']), TaggedDocument(words=['lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly'], tags=['8443']), TaggedDocument(words=['used', 'handbags', 'wallets', 'used', 'handbags', 'wallets', 'used', 'handbags', 'wallets', 'used', 'handbags', 'wallets'], tags=['6309'])]


In [19]:
model.train(sentences, total_examples=model.corpus_count, epochs=num_epochs)

In [20]:
model_name = 'doc2vec_hs04_prepro'
time_now = datetime.now().strftime('%d%m%Y_%H%M%S')

model_save_path = f'models/doc2vec/{model_name}_{num_epochs}epochs_{time_now}.d2v'
model.save(model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to models/doc2vec/doc2vec_hs04_prepro_50epochs_16112025_100706.d2v


In [21]:
del model

#### C- Preproced + N-gram descriptions

In [22]:
model = Doc2Vec(window=window, 
                min_count=1, # ignore 1 instead of not ignore any words
                vector_size=model_dim, # dim of the feature vectors
                sample=1e-4, # threshold randomly down-sample high-frequency words
                hs=1, # hierarchical softmax instead of negative sampling
                max_vocab_size=None, # no limit
                alpha=0.025, # initial learning rate
                min_alpha=0.001, # min learning rate
                dm=0, # PV-DBOW 
                dbow_words=0, # only trains doc-vectors
                dm_tag_count=1, # one tag per document
                dm_mean=0, # use the sum of the context word vectors
                dm_concat=0, # for smaller model
                negative=5, # number of negative samples
                seed=32, # random seed
                workers=os.cpu_count()
                )

In [23]:
print("Building vocabulary...")

sentences = []
for idx, row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    words_list = row[prepro_col].split() + row[ngram_col].split()
    sentences.append(TaggedDocument(words_list, [row[target_col]]))

print(sentences[:3])

model.build_vocab(sentences)

Building vocabulary...


100%|██████████| 242289/242289 [00:06<00:00, 36262.87it/s]


[TaggedDocument(words=['brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', 'x', 'ml', 'brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', '4', '50x200ml', 'brake_fluid', 'fluid_dot', 'dot_4', '4_50x200ml', '50x200ml_brake', 'brake_fluid', 'fluid_dot', 'dot_x', 'x_ml', 'ml_brake', 'brake_fluid', 'fluid_dot', 'dot_4', '4_50x200ml', '50x200ml_brake', 'brake_fluid', 'fluid_dot', 'dot_4', '4_50x200ml'], tags=['2710']), TaggedDocument(words=['lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly', 'lcd_assembly', 'assembly_lcd', 'lcd_assembly', 'assembly_lcd', 'lcd_assembly', 'assembly_lcd', 'lcd_assembly'], tags=['8443']), TaggedDocument(words=['used', 'handbags', 'wallets', 'used', 'handbags', 'wallets', 'used', 'handbags', 'wallets', 'used', 'handbags', 'wallets', 'used_handbags', 'handbags_wallets', 'wallets_used', 'used_handbags', 'handbags_wallets', 'wallets_used', 'used_handbags', 'handbags_wallets', 'wallets_used', 'used_handbags', 'ha

In [24]:
model.train(sentences, total_examples=model.corpus_count, epochs=num_epochs)

In [25]:

model_name = 'doc2vec_hs04_prepro+ngram'
time_now = datetime.now().strftime('%d%m%Y_%H%M%S')

model_save_path = f'models/doc2vec/{model_name}_{num_epochs}epochs_{time_now}.d2v'
model.save(model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to models/doc2vec/doc2vec_hs04_prepro+ngram_50epochs_16112025_102027.d2v


In [26]:
del model

## Training a FastText as baseline

A- Raw descriptions

B- Preproced descriptions

C- Preproced + N-gram descriptions

In [27]:
# # Model dependences
from gensim.models import FastText

target_col = 'HS04'
window = 5 # context window size +/- 
num_epochs = 50
model_dim = 254
seed = 32

raw_col = 'GOODS_DESCRIPTION'
prepro_col = 'PREPRO_DESCRIPTION'
ngram_col = 'NGRAM_DESCRIPTION'

### FastText configuration


FastText configuration to be used as Doc2Vec

In [28]:
class _DVShim:
    """Mimic gensim.Doc2Vec .dv interface with a minimal .most_similar(vectors, topn)."""
    def __init__(self, outer: "FastTextDocVec"):
        self.outer = outer

    def most_similar(self, vectors: List[np.ndarray], topn: int = 10):
        if isinstance(vectors, (list, tuple)):
            if len(vectors) != 1:
                raise ValueError("DVShim expects a single vector in a list like [vector].")
            inferred = np.asarray(vectors[0], dtype=np.float32)
        else:
            inferred = np.asarray(vectors, dtype=np.float32)
        return self.outer.most_similar(inferred, topn=topn)

class FastTextDocVec:
    """
    Minimal Doc2Vec-like API over FastText:
      - fit(df, text_col, label_col) trains FastText and caches doc vectors (L2-normalized means of word vectors)
      - infer_vector(text) -> np.ndarray (L2-normalized)
      - most_similar(vec, topn) -> [(tag, cosine), ...]
      - predict_classes(text, topn, k_neighbors) -> neighbor-vote by label

    Compatibility for your Doc2Vec evaluation:
      - .dv is a shim exposing .most_similar([vec], topn)
      - infer_vector(text) available
    """
    def __init__(self, dim: int = 254, window: int = 5, min_count: int = 3,
                 epochs: int = 20, sg: int = 1, min_n: int = 3, max_n: int = 6):
        
        self.dim = dim # dim of the feature vectors
        self.window = window # context window size +/-
        self.min_count = min_count # 0 = not ignore any words
        self.epochs = epochs
        self.sg = sg # 1 = skip-gram for similarity with doc2vec
        self.min_n = min_n
        self.max_n = max_n

        self.model: Optional[FastText] = None
        self.docvecs: Optional[np.ndarray] = None   # (n_docs, dim), L2-normalized
        self.tags: Optional[list[str]] = None       # list of doc ids aligned with docvecs
        self.labels: Optional[np.ndarray] = None    # array[str] aligned with docvecs

        # doc2vec-compat shim
        self.dv = _DVShim(self)

        # metadata (filled in fit)
        self._text_col = None
        self._label_col = None

    @staticmethod
    def _tokens(s: str) -> list:
        return s.split() if isinstance(s, str) and s.strip() else []
    
    def _docvec(self, tokens: list) -> np.ndarray:
        vecs = [self.model.wv[w] for w in tokens if w in self.model.wv]
        if not vecs:
            return np.zeros(self.dim, dtype=np.float32)
        v = np.mean(vecs, axis=0)
        n = np.linalg.norm(v) + 1e-12
        return (v / n).astype(np.float32)

    def fit(self, df, text_col: str = "GOODS_DESCRIPTION", label_col: str = "HS04"):
        """
        Train FastText robustly (explicit build_vocab/train) and cache:
        - self.docvecs: L2-normalized mean of word vectors per document
        - self.labels:  label per document (as str)
        - self.tags:    row index (as str), doc2vec-like
        """
        self._text_col, self._label_col = text_col, label_col

        # 1) Tokenize
        sentences = [self._tokens(x) for x in df[text_col].astype(str).tolist()]

        # Defensive: drop totally empty docs (avoid zero-only corpora)
        nonempty = [i for i, toks in enumerate(sentences) if len(toks) > 0]
        if not nonempty:
            raise ValueError("All documents are empty after tokenization/preprocessing.")
        if len(nonempty) < len(sentences):
            # Filter df and sentences together to avoid misalignment
            df = df.iloc[nonempty].copy()
            sentences = [sentences[i] for i in nonempty]

        # 2) Build model
        import os
        # - workers: set to max(1, os.cpu_count()-1) for portability (avoid -1)
        workers = max(1, (os.cpu_count() or 2) - 1)
        # - min_count: enforce >=1
        min_count = max(1, self.min_count)

        self.model = FastText(
            vector_size=self.dim,
            window=self.window,
            min_count=min_count,
            sg=self.sg,
            min_n=self.min_n,
            max_n=self.max_n,
            workers=workers
        )
        self.model.build_vocab(corpus_iterable=sentences)
        self.model.train(
            corpus_iterable=sentences,
            total_examples=len(sentences),
            epochs=self.epochs
        )

        # 3) Cache doc vectors
        self.docvecs = np.vstack([self._docvec(toks) for toks in sentences]).astype(np.float32)
        self.tags   = list(df.index.astype(str))
        self.labels = df[label_col].astype(str).to_numpy()

        # Diagnostics: % of zero vectors
        norms = np.linalg.norm(self.docvecs, axis=1)
        zero_ratio = float((norms < 1e-9).mean())
        if zero_ratio > 0.25:
            print(f"[WARN] {zero_ratio:.1%} of doc vectors are ~zero; check preprocessing/min_count.")

    def infer_vector(self, text: str) -> np.ndarray:
        toks = self._tokens(text)
        return self._docvec(toks)

    @staticmethod
    def _cosine(a: np.ndarray, b: np.ndarray) -> np.ndarray:
        # a: (d,), b: (n,d) -> (n,)
        a = a.reshape(1, -1).astype(np.float32, copy=False)
        a_norm = np.linalg.norm(a, axis=1, keepdims=True) + 1e-12
        b_norm = np.linalg.norm(b, axis=1, keepdims=True) + 1e-12
        return (a @ b.T).ravel() / (a_norm.ravel() * b_norm.ravel())

    def most_similar(self, inferred: np.ndarray, topn: int = 10) -> List[Tuple[str, float]]:
        sims = self._cosine(inferred, self.docvecs)
        k = min(topn, len(sims))
        idx = np.argpartition(-sims, k-1)[:k]
        ranked = sorted(((self.tags[i], float(sims[i])) for i in idx), key=lambda x: x[1], reverse=True)
        return ranked[:topn]

    def predict_classes(self, text: str, topn: int = 5, k_neighbors: int = 50) -> List[Tuple[str, float]]:
        inferred = self.infer_vector(text)
        sims = self._cosine(inferred, self.docvecs)
        k = min(k_neighbors, sims.shape[0])
        nn_idx = np.argpartition(-sims, k-1)[:k]
        agg = defaultdict(float)
        for i in nn_idx:
            agg[self.labels[i]] += float(sims[i])
        ranked = sorted(agg.items(), key=lambda x: x[1], reverse=True)
        return ranked[:topn]
    
    def save(self, directory: Union[str, Path]) -> None:
        """
        Save model + cached matrices and metadata to a folder:
           - fasttext.model (gensim)
           - docvecs.npy, labels.npy, tags.npy
           - meta.json  (dims, training params, columns)
        """
        directory = Path(directory)
        directory.mkdir(parents=True, exist_ok=True)

        assert self.model is not None, "Model is empty. Train before saving."
        assert self.docvecs is not None and self.labels is not None and self.tags is not None, \
            "Missing cached arrays. Train before saving."

        self.model.save(str(directory / "fasttext.model"))
        np.save(directory / "docvecs.npy", self.docvecs)
        np.save(directory / "labels.npy", self.labels)
        np.save(directory / "tags.npy", np.array(self.tags, dtype=object))

        meta = {
            "dim": self.dim, "window": self.window, "min_count": self.min_count,
            "epochs": self.epochs, "sg": self.sg, "min_n": self.min_n, "max_n": self.max_n,
            "text_col": self._text_col, "label_col": self._label_col
        }
        (directory / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, directory: Union[str, Path]) -> "FastTextDocVec":
        """Load a previously saved wrapper."""
        directory = Path(directory)
        meta = json.loads((directory / "meta.json").read_text(encoding="utf-8"))
        obj = cls(dim=meta["dim"], window=meta["window"], min_count=meta["min_count"],
                  epochs=meta["epochs"], sg=meta["sg"], min_n=meta["min_n"], max_n=meta["max_n"])
        obj._text_col = meta.get("text_col")
        obj._label_col = meta.get("label_col")

        obj.model = FastText.load(str(directory / "fasttext.model"))
        obj.docvecs = np.load(directory / "docvecs.npy")
        obj.labels = np.load(directory / "labels.npy", allow_pickle=True)
        obj.tags = np.load(directory / "tags.npy", allow_pickle=True).tolist()
        obj.dv = _DVShim(obj)   # reattach shim
        return obj

#### A- Raw descriptions

In [29]:
model = FastTextDocVec(dim=model_dim, 
                       window=window, 
                       min_count=1, # ignore 1 instead of not ignore any words 
                       epochs=num_epochs, 
                       sg=1, 
                       min_n=3, 
                       max_n=6)

model.fit(train_df, text_col=raw_col, label_col=target_col)

In [30]:
model_name = 'fasttext_hs04_raw'
time_now = datetime.now().strftime('%d%m%Y_%H%M%S')

folder_path = Path(f"models/fasttext/{model_name}")
model.save(folder_path)

print(f"Model saved in folder {folder_path} at {time_now}")

Model saved in folder models\fasttext\fasttext_hs04_raw at 16112025_102358


In [31]:
del model

#### B- Preproced descriptions

In [32]:
model = FastTextDocVec(dim=model_dim, 
                       window=window, 
                       min_count=1, # ignore 1 instead of not ignore any words 
                       epochs=num_epochs, 
                       sg=1, 
                       min_n=3, 
                       max_n=6)

model.fit(train_df, text_col=prepro_col, label_col=target_col)

In [33]:
model_name = 'fasttext_hs04_prepro'
time_now = datetime.now().strftime('%d%m%Y_%H%M%S')

folder_path = Path(f"models/fasttext/{model_name}")
model.save(folder_path)

print(f"Model saved in folder {folder_path} at {time_now}")

Model saved in folder models\fasttext\fasttext_hs04_prepro at 16112025_103340


In [34]:
del model

#### C- Preproced + N-gram descriptions

In [35]:
model = FastTextDocVec(dim=model_dim, 
                       window=window, 
                       min_count=1, # ignore 1 instead of not ignore any words 
                       epochs=num_epochs, 
                       sg=1, 
                       min_n=3, 
                       max_n=6)

train_df['prepro+ngram'] = (train_df[prepro_col] + ' ' + train_df[ngram_col]).str.strip()

model.fit(train_df, text_col='prepro+ngram', label_col=target_col)

In [36]:
model_name = 'fasttext_hs04_prepro+ngram'
time_now = datetime.now().strftime('%d%m%Y_%H%M%S')

folder_path = Path(f"models/fasttext/{model_name}")

model.save(folder_path)

print(f"Model saved in folder {folder_path} at {time_now}")

Model saved in folder models\fasttext\fasttext_hs04_prepro+ngram at 16112025_110510


In [37]:
del model

## Models evaluation

Testing dataset

In [38]:
val_df.head()

,HS06,GOODS_DESCRIPTION,HS04,HS02,PREPRO_DESCRIPTION,NGRAM_DESCRIPTION
11112,711719,USED EARRINGS,7117,71,used earrings used earrings used earrings used...,used_earrings earrings_used used_earrings earr...
161693,940360,BUREAU,9403,94,bureau bureau bureau bureau,bureau_bureau bureau_bureau bureau_bureau
83856,392290,SIDE PANEL SUITABLE FOR FONTE150/ FONTE 170 WI...,3922,39,side panel suitable fonte150/ fonte 170 fixing...,side_panel panel_suitable suitable_fonte150/ f...
329014,870323,USED TOYOTA RAV4 CHS NO: ZCA26-0051664 YEAR: 2004,8703,87,used toyota rav4 chs no: zca26-0051664 year: 2...,used_toyota toyota_rav4 rav4_chs chs_no: no:_z...
257684,961610,MULTI PURPOSE SPRAY (- SPRAYER TRIGGER),9616,96,multi purpose spray (- sprayer trigger) multi ...,multi_purpose purpose_spray spray_(- (-_spraye...


### Evaluating Doc2Vec

Evaluation function

In [39]:
def _to_tokens(x: Union[str, Iterable[str]]) -> List[str]:
    """
    Accepts either a raw string or an iterable of tokens and returns a token list.
    """
    if isinstance(x, str):
        x = x.strip()
        return x.split() if x else []
    if isinstance(x, Iterable):
        return [str(t) for t in x]
    raise TypeError(f"input_text must be str or iterable of str, not {type(x)}")


def predict_d2v(
    input_text: Union[str, Iterable[str]],
    model: Doc2Vec,
    top_n: int = 5,
    *,
    epochs: int = 30,
    alpha: Optional[float] = None,
    min_alpha: Optional[float] = None,
) -> List[Tuple[str, float]]:
    """
    Infer a vector for input_text and return top_n most similar documents (tag, score).
    Works with gensim>=4 (uses model.dv).
    """
    tokens = _to_tokens(input_text)
    # Safe guard: empty token list -> return empty results
    if not tokens:
        return []

    # infer_vector in gensim 4: epochs replaces steps; alpha/min_alpha optional
    inferred = model.infer_vector(tokens, epochs=epochs, alpha=alpha, min_alpha=min_alpha)

    # dv is the doc vectors (alias to docvecs in older gensim)
    dv = getattr(model, "dv", getattr(model, "docvecs", None))
    if dv is None:
        raise AttributeError("Doc2Vec model missing .dv/.docvecs")

    sims = dv.most_similar([inferred], topn=top_n)
    # Round ONLY for display; keep numeric float
    return [(str(tag), float(score)) for tag, score in sims]


def evaluate_df_d2v(
    val_df: pd.DataFrame,
    model_name: str = "",
    model_path: str = "",
    *,
    text_col: str = "GOODS_DESCRIPTION",
    target_col: str = "HS04",
    top_n: int = 5,
    epochs: int = 30,
    alpha: Optional[float] = None,
    min_alpha: Optional[float] = None,
    show_progress: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """
    For each row, run predict_d2v and create top_k prediction columns: top_1..top_k and top_1_SCORE..top_k_SCORE.
    Returns (scored_df, metrics) with top-k accuracies.
    """
    print(f"# Evaluating model: {model_name}")
    print(f"Text column: {text_col}")
    print(f"Target column: {target_col}")
    print(f"Top-N: {top_n}")

    # Load model
    model = Doc2Vec.load(model_path)
    print(f"Model loaded from: {model_path}")

    df_ = val_df.copy()

    # Ensure target is string for fair comparison (HS codes often have leading zeros)
    if target_col in df_.columns:
        df_[target_col] = df_[target_col].astype(str)

    # Containers for predictions
    preds_tags = [[] for _ in range(top_n)]
    preds_scores = [[] for _ in range(top_n)]

    iterator = df_.itertuples(index=False, name=None)
    if show_progress:
        iterator = tqdm(iterator, total=df_.shape[0])

    # Map column index for speed in itertuples
    cols = list(df_.columns)
    text_idx = cols.index(text_col)

    for row in iterator:
        text_val = row[text_idx]
        try:
            sims = predict_d2v(
                text_val, model, top_n=top_n, epochs=epochs, alpha=alpha, min_alpha=min_alpha
            )
        except Exception as e:
            # In case of any unexpected row-level issue, fill with nulls and continue
            sims = []

        # Normalize to length top_n with blanks if fewer results (e.g., empty text)
        if len(sims) < top_n:
            sims = sims + [("", float("nan"))] * (top_n - len(sims))

        for k in range(top_n):
            tag, score = sims[k]
            preds_tags[k].append(tag)
            preds_scores[k].append(score)

    # Write columns programmatically
    for k in range(top_n):
        df_[f"top_{k+1}"] = preds_tags[k]
        df_[f"top_{k+1}_SCORE"] = preds_scores[k]

    # ----- Metrics (top-k accuracy) -----
    metrics: Dict[str, float] = {}
    if target_col in df_.columns:
        y_true = df_[target_col].astype(str)
        # We compute cumulative "any correct up to k"
        hit_cumulative = None
        for k in range(1, top_n + 1):
            eq_k = y_true.eq(df_[f"top_{k}"].astype(str))
            hit_cumulative = eq_k if hit_cumulative is None else (hit_cumulative | eq_k)
            acc_k = float(hit_cumulative.mean())
            metrics[f"top_{k}_acc"] = round(acc_k, 6)

        total = df_.shape[0]
        print(f"Total samples: {total}")
        for k in range(1, top_n + 1):
            correct = int(metrics[f'top_{k}_acc'] * total)
            print(f"Top-{k} Accuracy: {metrics[f'top_{k}_acc']:.4f} ({correct}/{total})")
    else:
        print("(No target column present; metrics skipped.)")
        

    return df_, metrics


In [40]:
raw_col = 'GOODS_DESCRIPTION'
prepro_col = 'PREPRO_DESCRIPTION'
ngram_col = 'NGRAM_DESCRIPTION'
target_col = 'HS04'   # make it explicit

val_df = val_df.copy()
for c in [raw_col, prepro_col, ngram_col]:
    if c in val_df.columns:
        val_df[c] = val_df[c].fillna('').astype(str)
    else:
        raise KeyError(f"Missing expected column: {c}")

val_df['prepro+ngram'] = (val_df[prepro_col] + ' ' + val_df[ngram_col]).str.strip()

models = [
    ('doc2vec_hs04_raw',
     'models/doc2vec/doc2vec_hs04_raw_50epochs_16112025_100112.d2v',
     raw_col, target_col),

    ('doc2vec_hs04_prepro',
     'models/doc2vec/doc2vec_hs04_prepro_50epochs_16112025_100706.d2v',
     prepro_col, target_col),

    ('doc2vec_hs04_prepro+ngram',
     'models/doc2vec/doc2vec_hs04_prepro+ngram_50epochs_16112025_102027.d2v',
     'prepro+ngram', target_col),
]

all_metrics = []
scored_dfs = {}

for model_name, model_path, text_col, tgt in models:
    df_scored, metrics = evaluate_df_d2v(
        val_df,
        model_name=model_name,
        model_path=model_path,
        text_col=text_col,
        target_col=tgt,
        top_n=5,
        epochs=num_epochs,           # matches your trained-model naming
        alpha=None,          # let gensim handle the schedule
        min_alpha=None,
        show_progress=True
    )
    scored_dfs[model_name] = df_scored
    row = {'model': model_name, **metrics}
    all_metrics.append(row)

metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
print("\n=== Metrics summary ===")
metrics_df

# Evaluating model: doc2vec_hs04_raw
Text column: GOODS_DESCRIPTION
Target column: HS04
Top-N: 5
Model loaded from: models/doc2vec/doc2vec_hs04_raw_50epochs_16112025_100112.d2v


  0%|          | 0/26778 [00:00<?, ?it/s]C:\Users\santt\AppData\Local\Temp\ipykernel_38520\2712469892.py:35: DeprecationWarning: Call to deprecated `docvecs` (The `docvecs` property has been renamed `dv`.).
  dv = getattr(model, "dv", getattr(model, "docvecs", None))
100%|██████████| 26778/26778 [00:31<00:00, 838.85it/s]


Total samples: 26778
Top-1 Accuracy: 0.5073 (13584/26778)
Top-2 Accuracy: 0.5987 (16031/26778)
Top-3 Accuracy: 0.6432 (17224/26778)
Top-4 Accuracy: 0.6722 (18000/26778)
Top-5 Accuracy: 0.6916 (18521/26778)
# Evaluating model: doc2vec_hs04_prepro
Text column: PREPRO_DESCRIPTION
Target column: HS04
Top-N: 5
Model loaded from: models/doc2vec/doc2vec_hs04_prepro_50epochs_16112025_100706.d2v


  0%|          | 0/26778 [00:00<?, ?it/s]C:\Users\santt\AppData\Local\Temp\ipykernel_38520\2712469892.py:35: DeprecationWarning: Call to deprecated `docvecs` (The `docvecs` property has been renamed `dv`.).
  dv = getattr(model, "dv", getattr(model, "docvecs", None))
100%|██████████| 26778/26778 [01:12<00:00, 370.80it/s]


Total samples: 26778
Top-1 Accuracy: 0.4662 (12482/26778)
Top-2 Accuracy: 0.5450 (14594/26778)
Top-3 Accuracy: 0.5850 (15664/26778)
Top-4 Accuracy: 0.6115 (16374/26778)
Top-5 Accuracy: 0.6315 (16910/26778)
# Evaluating model: doc2vec_hs04_prepro+ngram
Text column: prepro+ngram
Target column: HS04
Top-N: 5
Model loaded from: models/doc2vec/doc2vec_hs04_prepro+ngram_50epochs_16112025_102027.d2v


  0%|          | 0/26778 [00:00<?, ?it/s]C:\Users\santt\AppData\Local\Temp\ipykernel_38520\2712469892.py:35: DeprecationWarning: Call to deprecated `docvecs` (The `docvecs` property has been renamed `dv`.).
  dv = getattr(model, "dv", getattr(model, "docvecs", None))
100%|██████████| 26778/26778 [02:48<00:00, 158.80it/s]


Total samples: 26778
Top-1 Accuracy: 0.6108 (16355/26778)
Top-2 Accuracy: 0.6872 (18402/26778)
Top-3 Accuracy: 0.7154 (19157/26778)
Top-4 Accuracy: 0.7324 (19611/26778)
Top-5 Accuracy: 0.7437 (19916/26778)

=== Metrics summary ===


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
model,,,,,
doc2vec_hs04_prepro,0.466166,0.545000,0.584958,0.611509,0.631489
doc2vec_hs04_prepro+ngram,0.610763,0.687206,0.715438,0.732392,0.743745
doc2vec_hs04_raw,0.507319,0.598700,0.643215,0.672194,0.691650


### Evaluating FastText

Evaluation function

In [ ]:
def predict_ft(
    input_text: Union[str, Iterable[str]],
    model,                     # FastTextDocVec instance OR dict-like wrapper
    top_n: int = 5,
    *,
    epochs: int = 30,          # ignored (for signature parity)
    alpha: Optional[float] = None,      # ignored
    min_alpha: Optional[float] = None,  # ignored
) -> List[Tuple[str, float]]:
    """
    Infer a vector and return top_n (HS04_label, score) pairs.
    Uses model.predict_classes if available; otherwise maps most-similar tags to labels.
    """
    tokens = _to_tokens(input_text)
    if not tokens:
        return []

    text = " ".join(tokens)

    # 1) Preferred path: wrapper exposes label-level predictor
    if hasattr(model, "predict_classes"):
        # many wrappers default k_neighbors internally (often 50)
        preds = model.predict_classes(text, topn=top_n)
        # Ensure (label, score) as (str, float)
        return [(str(lbl), float(scr)) for (lbl, scr) in preds]

    # 2) Fallback: infer vector + nearest docs (tags), then map tag -> label if possible
    infer_vector = None
    dv = None

    if isinstance(model, dict):
        infer_vector = model.get("infer_vector", None)
        dv = model.get("dv", None)
    else:
        # some wrappers expose these as attributes
        infer_vector = getattr(model, "infer_vector", None)
        dv = getattr(model, "dv", None)

    if infer_vector is None or dv is None:
        return []

    inferred = infer_vector(text)
    sims = dv.most_similar([inferred], topn=top_n)  # [(tag, score)]

    # Try to map tag -> label
    tag_to_label = getattr(model, "tag_to_label", None)
    if tag_to_label and isinstance(tag_to_label, dict):
        return [(str(tag_to_label.get(tag, "")), float(score)) for tag, score in sims]

    # Or map via parallel arrays 'tags' and 'labels'
    tags = getattr(model, "tags", None)
    labels = getattr(model, "labels", None)
    if tags is not None and labels is not None and len(tags) == len(labels):
        pos = {t: i for i, t in enumerate(tags)}
        out = []
        for tag, score in sims:
            i = pos.get(tag, None)
            out.append((str(labels[i]) if i is not None else "", float(score)))
        return out

    # If we cannot map, return empty to avoid polluting metrics
    return []


def evaluate_df_ft(
    val_df: pd.DataFrame,
    model_name: str = "",
    model_path: str = "",     # directory produced by FTDoc2VecLike.save()
    *,
    text_col: str = "GOODS_DESCRIPTION",
    target_col: str = "HS04",
    top_n: int = 5,
    epochs: int = 30,                 # accepted for signature parity (ignored)
    alpha: Optional[float] = None,    # ignored
    min_alpha: Optional[float] = None,# ignored
    show_progress: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """
    Same API/outputs as before.
    Minimal fixes:
      - call predict_ft (not the model object)
      - ensure predict_ft returns HS04 labels, not tags
      - trim memory: evaluate on a slim copy; downcast scores to float32
    """
    print(f"# Evaluating model: {model_name}")
    print(f"Text column: {text_col}")
    print(f"Target column: {target_col}")
    print(f"Top-N: {top_n}")

    ft = FastTextDocVec.load(model_path)  # avoid shadowing 'model' name
    print(f"Model loaded from: {model_path}")

    # work on a slim copy to save RAM
    base_cols = [c for c in (text_col, target_col) if c in val_df.columns]
    df_ = val_df[base_cols].copy()

    if target_col in df_.columns:
        df_[target_col] = df_[target_col].astype(str)

    preds_tags = [[] for _ in range(top_n)]
    preds_scores = [[] for _ in range(top_n)]

    iterator = df_.itertuples(index=False, name=None)
    if show_progress:
        iterator = tqdm(iterator, total=df_.shape[0])

    cols = list(df_.columns)
    text_idx = cols.index(text_col)

    for row in iterator:
        text_val = row[text_idx]
        try:
            sims = predict_ft(text_val, ft, top_n=top_n) 
        except Exception:
            sims = []

        if len(sims) < top_n:
            sims = sims + [("", float("nan"))] * (top_n - len(sims))

        for k in range(top_n):
            tag, score = sims[k]
            preds_tags[k].append("" if tag is None else str(tag))
            # downcast to float32 to cut memory
            try:
                preds_scores[k].append(np.float32(score))
            except Exception:
                preds_scores[k].append(np.float32(np.nan))

    del ft

    # attach predictions (keep dtypes lean)
    for k in range(top_n):
        df_[f"top_{k+1}"] = pd.Series(preds_tags[k], index=df_.index, dtype="string")
        df_[f"top_{k+1}_SCORE"] = pd.Series(preds_scores[k], index=df_.index, dtype="float32")

    metrics: Dict[str, float] = {}
    if target_col in df_.columns:
        y_true = df_[target_col].astype(str)
        hit_cum = None
        for k in range(1, top_n + 1):
            eq_k = y_true.eq(df_[f"top_{k}"].astype(str))
            hit_cum = eq_k if hit_cum is None else (hit_cum | eq_k)
            acc_k = float(hit_cum.mean())
            metrics[f"top_{k}_acc"] = round(acc_k, 6)

        total = df_.shape[0]
        print(f"Total samples: {total}")
        for k in range(1, top_n + 1):
            correct = int(metrics[f"top_{k}_acc"] * total)
            print(f"Top-{k} Accuracy: {metrics[f'top_{k}_acc']:.4f} ({correct}/{total})")
    else:
        print("(No target column present; metrics skipped.)")

    return df_, metrics


In [ ]:
raw_col = 'GOODS_DESCRIPTION'
prepro_col = 'PREPRO_DESCRIPTION'
ngram_col = 'NGRAM_DESCRIPTION'
target_col = 'HS04'
TOP_N      = 5
KNN        = 50

val_df = val_df.copy()
for c in [raw_col, prepro_col, ngram_col]:
    if c in val_df.columns:
        val_df[c] = val_df[c].fillna('').astype(str)
    else:
        raise KeyError(f"Missing expected column: {c}")

val_df['prepro+ngram'] = (val_df[prepro_col] + ' ' + val_df[ngram_col]).str.strip()

# Point to the saved FT wrapper directories
ft_models = [
    ('fasttext_hs04_raw',           'models/fasttext/fasttext_hs04_raw',            raw_col, target_col),
    ('fasttext_hs04_prepro',        'models/fasttext/fasttext_hs04_prepro',         prepro_col, target_col),
    ('fasttext_hs04_prepro+ngram',  'models/fasttext/fasttext_hs04_prepro+ngram',   'prepro+ngram', target_col),
]

all_metrics = []
scored_dfs = {}

for model_name, model_path, text_col, tgt in ft_models:
    df_scored, metrics = evaluate_df_ft(
        val_df,
        model_name=model_name,
        model_path=model_path,
        text_col=text_col,
        target_col=tgt,
        top_n=5,
        epochs=num_epochs,
        alpha=None,
        min_alpha=None,
        show_progress=True
    )
    scored_dfs[model_name] = df_scored
    all_metrics.append({'model': model_name, **metrics})

metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
print("\n=== Metrics summary (FastText Doc2Vec-like) ===")
display(metrics_df)

# Evaluating model: fasttext_hs04_raw
Text column: GOODS_DESCRIPTION
Target column: HS04
Top-N: 5
Model loaded from: models/fasttext/fasttext_hs04_raw


100%|██████████| 26778/26778 [45:11<00:00,  9.88it/s]


Total samples: 26778
Top-1 Accuracy: 0.5336 (14289/26778)
Top-2 Accuracy: 0.6583 (17629/26778)
Top-3 Accuracy: 0.7168 (19195/26778)
Top-4 Accuracy: 0.7516 (20125/26778)
Top-5 Accuracy: 0.7753 (20761/26778)
# Evaluating model: fasttext_hs04_prepro
Text column: PREPRO_DESCRIPTION
Target column: HS04
Top-N: 5


In [43]:
df_scored  

,HS06,GOODS_DESCRIPTION,HS04,HS02,PREPRO_DESCRIPTION,NGRAM_DESCRIPTION,prepro+ngram,top_1,top_1_SCORE,top_2,top_2_SCORE,top_3,top_3_SCORE,top_4,top_4_SCORE,top_5,top_5_SCORE
11112,711719,USED EARRINGS,7117,71,used earrings used earrings used earrings used...,used_earrings earrings_used used_earrings earr...,used earrings used earrings used earrings used...,,NaN,,NaN,,NaN,,NaN,,NaN
161693,940360,BUREAU,9403,94,bureau bureau bureau bureau,bureau_bureau bureau_bureau bureau_bureau,bureau bureau bureau bureau bureau_bureau bure...,,NaN,,NaN,,NaN,,NaN,,NaN
83856,392290,SIDE PANEL SUITABLE FOR FONTE150/ FONTE 170 WI...,3922,39,side panel suitable fonte150/ fonte 170 fixing...,side_panel panel_suitable suitable_fonte150/ f...,side panel suitable fonte150/ fonte 170 fixing...,,NaN,,NaN,,NaN,,NaN,,NaN
329014,870323,USED TOYOTA RAV4 CHS NO: ZCA26-0051664 YEAR: 2004,8703,87,used toyota rav4 chs no: zca26-0051664 year: 2...,used_toyota toyota_rav4 rav4_chs chs_no: no:_z...,used toyota rav4 chs no: zca26-0051664 year: 2...,,NaN,,NaN,,NaN,,NaN,,NaN
257684,961610,MULTI PURPOSE SPRAY (- SPRAYER TRIGGER),9616,96,multi purpose spray (- sprayer trigger) multi ...,multi_purpose purpose_spray spray_(- (-_spraye...,multi purpose spray (- sprayer trigger) multi ...,,NaN,,NaN,,NaN,,NaN,,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76008,392350,"HOTPAK PAPER JUICE CUP12OZ, HOTPACK, PJC12BHP,...",3923,39,"hotpak paper juice cup12oz, hotpack, pjc12bhp,...","hotpak_paper paper_juice juice_cup12oz, cup12o...","hotpak paper juice cup12oz, hotpack, pjc12bhp,...",,NaN,,NaN,,NaN,,NaN,,NaN
330681,732393,A1153 ProLine Frypan w/out Lid 24x5cm,7323,73,a1153 proline frypan w/out lid 24x5cm a pr...,a1153_proline proline_frypan frypan_w/out w/ou...,a1153 proline frypan w/out lid 24x5cm a pr...,,NaN,,NaN,,NaN,,NaN,,NaN
79812,761010,ALUMINIUM SLIDING DOOR,7610,76,aluminium sliding door aluminium sliding door ...,aluminium_sliding sliding_door door_aluminium ...,aluminium sliding door aluminium sliding door ...,,NaN,,NaN,,NaN,,NaN,,NaN
448169,730799,P.V.C F/TEE,7307,73,p.v.c f/tee p v c f tee p v c f tee p.v.c f/tee,p.v.c_f/tee f/tee_p p_v v_c c_f f_tee tee_p p_...,p.v.c f/tee p v c f tee p v c f tee p.v.c f/te...,,NaN,,NaN,,NaN,,NaN,,NaN


In [44]:
# --- QUICK SANITY CHECKS (run after your current evaluation) ---

def _looks_like_hs4(s: str) -> bool:
    # crude HS04 check: exactly 4 digits
    import re
    return isinstance(s, str) and re.fullmatch(r"\d{4}", s or "") is not None

# Pick the scored dataframe you already created (e.g., df_scored)
_df = df_scored  # change if you use another variable name

# 1) Are predictions HS codes or document tags?
ex_pred = _df["top_1"].astype(str).head(20).tolist()
print("[sample top_1 values]", ex_pred)

ratio_looks_like_hs = _df["top_1"].astype(str).map(_looks_like_hs4).mean()
print(f"% of top_1 that look like HS04 codes: {ratio_looks_like_hs:.4f}")

# 2) Compare the universe of predicted tokens vs target labels
pred_unique = set(_df["top_1"].astype(str).unique()[:1000])
true_unique = set(_df["HS04"].astype(str).unique())
overlap = len(pred_unique & true_unique) / max(1, len(pred_unique))
print(f"Overlap (unique top_1 vs unique HS04): {overlap:.4f}")

# If ratio_looks_like_hs ~ 0 and overlap ~ 0 -> you're comparing tags to labels (root cause)

[sample top_1 values] ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
% of top_1 that look like HS04 codes: 0.0000
Overlap (unique top_1 vs unique HS04): 0.0000


## Observations



## Future steps

